# phase_2 detector on Kaggle — YOLOv8l, Drive-resumable

Trains the 29-region detector **exactly like the local run**, but checkpoints to
**Google Drive via rclone** so it survives Kaggle's ephemeral storage.

**Why Drive:** `/kaggle/working` is wiped between sessions unless you *Commit*, and
a session can die mid-run. We push the run dir to Drive **every `SYNC_EVERY`
optimizer steps + each epoch**; the next session pulls `last.pt` and `--resume`s.

**Each session:** Settings → Accelerator → **GPU**, then *Run All*. From the 2nd
session on, set `RESUME = 1` in CONFIG. That's the only change.

## CONFIG — edit these

In [ ]:
import os
# ===== code comes from GitHub (cloned in the next cell), not a code dataset =====
PHASE2_SRC = "/kaggle/working/repo/phase_2"
# ===== Kaggle dataset slugs (edit) =====
IMAGES      = "/kaggle/input/datasets/nguynnghin/mimic-cxr-448/mimic-cxr-448"  # 448 images (stem == image_id)
YOLO_LABELS = "/kaggle/input/datasets/nguynnghin/yolo-labels"   # PREBUILT labels (gold 784 ids skipped at link time)

# ===== training (OPTION A: fast warm-start run that fits one session) =====
# MODEL doubles as the INIT-WEIGHTS knob: keep "yolov8m.pt" for a fresh run, OR set it to a
# saved best.pt path to CONTINUE training from an earlier run (warm start, RESUME stays 0).
MODEL    = "yolov8m.pt"   # m (not l): ~2x faster, near-identical box quality for 29 easy regions
IMGSZ    = 448            # images are 448 (bump to 640 later if tiny regions like carina/svc drag mAP)
EPOCHS   = 30             # 29 regions converge fast; patience 15 cuts earlier if mAP plateaus
FRACTION = 0.25           # ~37k imgs: full data + 50ep only reached ~0.68, so data is NOT the lever
RUN_NAME = "det29"
RESUME   = 0             # 0 = fresh; 1 = continue an INTERRUPTED run from the Drive checkpoint

# ===== durable checkpoints (Google Drive via a SERVICE ACCOUNT) =====
# dhint: is defined entirely by the SA + the folder id below -> no rclone.conf, no token.
# Because root = your CHEX-DATA folder, paths DROP the 'CHEX-DATA/' prefix.
DRIVE_FOLDER_ID = "1c6AJ3fjsL449kiMK4xiXfnzwzA4gIo0O"   # CHEX-DATA folder id
RCLONE_REMOTE   = "dhint:phase2_runs"           # = CHEX-DATA/phase2_runs
SYNC_EVERY      = 300                            # push every N optimizer steps

for k, v in dict(PHASE2_SRC=PHASE2_SRC, IMAGES=IMAGES, YOLO_LABELS=YOLO_LABELS, MODEL=MODEL,
                 IMGSZ=IMGSZ, EPOCHS=EPOCHS, FRACTION=FRACTION, RUN_NAME=RUN_NAME, RESUME=RESUME,
                 DRIVE_FOLDER_ID=DRIVE_FOLDER_ID, RCLONE_REMOTE=RCLONE_REMOTE, SYNC_EVERY=SYNC_EVERY).items():
    os.environ[k] = str(v)
print("config set | MODEL =", MODEL, "| FRACTION =", FRACTION, "| EPOCHS =", EPOCHS, "| RESUME =", RESUME)

In [2]:
# --- get the code from GitHub (instead of uploading a code dataset). Internet ON. ---
!rm -rf /kaggle/working/repo && git clone -q https://github.com/hiennguyendang/phase_2_3_4_5 /kaggle/working/repo
!ls /kaggle/working/repo/phase_2 | head

build_pseudo_scene_graph.py
build_sft_dataset.py
build_yolo_dataset.py
config.py
constants.py
eval_yolo.py
extract_sg_vocab.py
finetune_sg_llm.py
infer_yolo.py
kaggle_sync.py


## 1 — install rclone + auth via YOUR Google account (OAuth token)

Service accounts have **no Drive storage quota** and CANNOT upload into a *My Drive*
folder (`403 storageQuotaExceeded`) — they can only read/mkdir. So we authenticate as
**you** with an OAuth token; files then upload to your own Drive (your quota).

**One-time setup, on a machine WITH a browser (your laptop):**
1. Install rclone — https://rclone.org/downloads/ or `winget install Rclone.Rclone`.
2. Run `rclone authorize "drive"` → a browser opens → log in with the Google account
   that **owns CHEX-DATA** → *Allow*.
3. rclone prints a token between `--->` and `<---End` — copy the whole `{...}` JSON.
4. Kaggle **Add-ons → Secrets → Add**: label **`GDRIVE_TOKEN`**, value = that `{...}`.
5. Keep the CHEX-DATA folder id in `DRIVE_FOLDER_ID` (CONFIG). Uploads land in
   CHEX-DATA/phase2_runs, charged to your quota. The cell below runs a real write-test.

In [3]:
%%bash
set -e
if ! command -v rclone >/dev/null 2>&1; then
  cd /kaggle/working
  curl -sLO https://downloads.rclone.org/rclone-current-linux-amd64.zip
  unzip -q -o rclone-current-linux-amd64.zip
  cp rclone-*-linux-amd64/rclone /usr/local/bin/ && chmod +x /usr/local/bin/rclone
fi
rclone version | head -1

rclone v1.74.3


In [ ]:
import os
# Drive remote 'dhint' via YOUR OAuth token (NOT a service account: SA has no storage quota
# and 403s on upload to My Drive). Token comes from `rclone authorize "drive"` -> Kaggle
# Secret GDRIVE_TOKEN. Graceful: if missing/unwritable, training still runs WITHOUT sync.
SYNC_OK = "0"
try:
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    token = sec.get_secret("GDRIVE_TOKEN").strip()          # the {...} from rclone authorize
    os.environ["RCLONE_CONFIG_DHINT_TYPE"] = "drive"
    os.environ["RCLONE_CONFIG_DHINT_TOKEN"] = token
    os.environ["RCLONE_CONFIG_DHINT_SCOPE"] = "drive"
    os.environ["RCLONE_CONFIG_DHINT_ROOT_FOLDER_ID"] = os.environ["DRIVE_FOLDER_ID"]  # = CHEX-DATA (yours)
    os.environ.pop("RCLONE_CONFIG_DHINT_SERVICE_ACCOUNT_FILE", None)  # drop stale SA from a prior cell run
    # OPTIONAL: your OWN Google API client -> private query quota (avoids rclone's shared-client
    # RATE_LIMIT_EXCEEDED on long runs). Set GDRIVE_CLIENT_ID/SECRET secrets + re-auth the token
    # with `rclone authorize "drive" "<id>" "<secret>"`. Absent -> rclone's shared client (works).
    for k_sec, k_env in [("GDRIVE_CLIENT_ID", "RCLONE_CONFIG_DHINT_CLIENT_ID"),
                         ("GDRIVE_CLIENT_SECRET", "RCLONE_CONFIG_DHINT_CLIENT_SECRET")]:
        try:
            os.environ[k_env] = sec.get_secret(k_sec).strip()
        except Exception:
            pass
    using_own = "own client" if os.environ.get("RCLONE_CONFIG_DHINT_CLIENT_ID") else "rclone shared client"
    remote = os.environ["RCLONE_REMOTE"]
    if os.system('rclone mkdir "%s"' % remote) == 0 and \
       os.system('echo ok | rclone rcat "%s/_write_test.txt"' % remote) == 0:
        # SA failed exactly at the upload step -> prove we can WRITE before training
        SYNC_OK = "1"
        os.system('rclone delete "%s/_write_test.txt" 2>/dev/null' % remote)
        print(f"remote OK (OAuth, write verified, {using_own}) ->", remote)
        os.system('rclone lsd dhint: 2>/dev/null | head')
    else:
        print("[WARN] Drive write FAILED -> check GDRIVE_TOKEN (own account, write scope) + DRIVE_FOLDER_ID")
except Exception as e:
    print("[WARN] GDRIVE_TOKEN secret not set -> NO Drive sync:", e)
os.environ["SYNC_OK"] = SYNC_OK
print("SYNC_OK =", SYNC_OK)

## 2 — copy code into the writable dir + install ultralytics

In [5]:
!rm -rf /kaggle/working/phase_2 && cp -r "$PHASE2_SRC" /kaggle/working/phase_2
%cd /kaggle/working/phase_2
!pip install -q ultralytics

/kaggle/working/phase_2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.8 MB/s eta 0:00:00


In [ ]:
# verify GPU BEFORE training (P100 = sm_60 is NOT supported by torch 2.10 -> use T4!)
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "| device_count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0), "| capability:", torch.cuda.get_device_capability(0))
assert torch.cuda.is_available(), "No GPU! Settings -> Accelerator -> GPU T4 x2, then Run All."
assert torch.cuda.get_device_capability(0) >= (7, 0), \
    "GPU too old for torch 2.10 (need sm_70+). P100 is sm_60 -> switch to T4."

## 3 — assemble the YOLO dataset (labels built LOCALLY)

The slow part (open every image for W/H + convert boxes) was done **once locally** with
`build_yolo_dataset.py --labels-only` and uploaded as the **`yolo-labels`** dataset. Here we just
**symlink the matching images** + write `dataset.yaml` — fast, no PIL, no path autodetect headaches.
So you no longer upload the 11 GB scene-graph dataset or the metadata to Kaggle at all.

In [ ]:
import os, glob, zipfile
YL = os.environ["YOLO_LABELS"]
# Kaggle keeps the upload as labels.zip (not auto-extracted) -> unzip once into /kaggle/working
zips = glob.glob(os.path.join(YL, "**", "labels.zip"), recursive=True)
LDIR = YL
if zips:
    LDIR = "/kaggle/working/yolo_labels_ex"
    os.makedirs(LDIR, exist_ok=True)
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(LDIR)
    print("unzipped", zips[0], "->", LDIR)
os.environ["LDIR"] = LDIR
!python link_yolo_images.py --labels-dir "$LDIR" --images-root "$IMAGES" --out /kaggle/working/yolo_ds

In [ ]:
# sanity: dataset.yaml + how many images/labels linked per split (gold 784 skipped from test)
import os, glob
DS = "/kaggle/working/yolo_ds"
print(open(f"{DS}/dataset.yaml").read())
for s in ("train", "val", "test"):
    imgs = glob.glob(f"{DS}/images/{s}/*")
    lbls = glob.glob(f"{DS}/labels/{s}/*")
    print(f"{s}: images={len(imgs)} labels={len(lbls)}")
sample = glob.glob(f"{DS}/images/train/*")[:1]
if sample:
    print("symlink ->", os.path.realpath(sample[0]))

## 4 — train (Drive-synced)

Fresh run pushes checkpoints to Drive every `SYNC_EVERY` steps + each epoch.
If the session dies, set `RESUME=1` in CONFIG and *Run All* again — `train_yolo.py`
pulls `last.pt` from Drive and continues.

In [ ]:
# background Drive sync (DDP-safe) — RUN THIS BEFORE the train cell.
# ultralytics DDP (--device 0,1) does NOT propagate model.add_callback hooks to its worker
# processes, so kaggle_sync's callbacks won't fire under 2 GPUs. This daemon thread is
# INDEPENDENT of them: it mirrors /kaggle/working/runs -> Drive every SYNC_SECONDS. It keeps
# running while the next cell's `!python train_yolo.py` blocks (threads run during the
# subprocess wait), so checkpoints reach Drive mid-run even with DDP.
import os, threading, subprocess

SYNC_SECONDS = 300
_bg_stop = threading.Event()

def _bg_sync():
    remote = os.environ["RCLONE_REMOTE"]          # dhint:phase2_runs
    runs = "/kaggle/working/runs"                  # contains det29/ -> lands at phase2_runs/det29
    while not _bg_stop.wait(SYNC_SECONDS):
        if os.path.isdir(runs):
            try:
                subprocess.run(["rclone", "copy", runs, remote,
                                "--transfers", "8", "--checkers", "8", "--quiet"],
                               check=False, timeout=600)
                print(f"[bg-sync] pushed runs/ -> {remote}", flush=True)
            except Exception as e:
                print(f"[bg-sync] push failed (continuing): {e}", flush=True)

if os.environ.get("SYNC_OK") == "1":
    threading.Thread(target=_bg_sync, daemon=True).start()
    print(f"[bg-sync] started: runs/ -> {os.environ['RCLONE_REMOTE']} every {SYNC_SECONDS}s")
else:
    print("[bg-sync] Drive off (SYNC_OK != 1) -> no background sync")

In [ ]:
import os
# 2-GPU DDP. Mid-run Drive sync is done by the BACKGROUND thread above (DDP drops the
# kaggle_sync callbacks). We still pass --sync-remote so a --resume can PULL last.pt from
# Drive first, but set --sync-every huge so the (non-firing) callback never double-pushes
# against the background thread. --batch is FIXED (autobatch -1 is unreliable under DDP);
# 32 total = 16 per T4.
SR = os.environ["RCLONE_REMOTE"] if os.environ.get("SYNC_OK") == "1" else ""
os.environ["SYNC"] = ("--sync-remote %s --sync-every 99999999" % SR) if SR else ""
os.environ["RESUME_FLAG"] = "--resume" if int(os.environ["RESUME"]) else ""
print("2-GPU DDP | bg-sync handles Drive | resume:", os.environ["RESUME_FLAG"] or "(fresh)")
!python train_yolo.py \
  --data /kaggle/working/yolo_ds/dataset.yaml \
  --model "$MODEL" --device 0,1 --imgsz $IMGSZ --epochs $EPOCHS --fraction $FRACTION --batch 32 \
  --runs /kaggle/working/runs --name "$RUN_NAME" $SYNC $RESUME_FLAG

## 5 — eval mAP (val; use `--split test` for the gold human-verified set)

In [ ]:
!python eval_yolo.py \
  --weights /kaggle/working/runs/$RUN_NAME/weights/best.pt \
  --data /kaggle/working/yolo_ds/dataset.yaml --split val --device 0

## 6 — grab the trained model

`best.pt` is already on Drive (final push at train end). To pull it anywhere:

```bash
rclone copy dhint:phase2_runs/det29/weights/best.pt ./   # dhint: root = CHEX-DATA
```
Or download it from the Kaggle notebook's **Output** tab (`runs/det29/weights/best.pt`).

In [ ]:
import os
RUN = os.environ["RUN_NAME"]
if os.environ.get("SYNC_OK") == "1":
    os.system('rclone copy /kaggle/working/runs/%s "%s/%s" --quiet' % (RUN, os.environ["RCLONE_REMOTE"], RUN))
    print("pushed final run -> %s/%s" % (os.environ["RCLONE_REMOTE"], RUN))
else:
    print("Drive sync off -> download from the notebook 'Output' tab: runs/%s/weights/best.pt" % RUN)
!ls -lh /kaggle/working/runs/$RUN_NAME/weights/